In [ ]:
import os
import glob
import pandas as pd
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests
import numpy as np
import sys
import warnings
warnings.filterwarnings("ignore")
import gzip
import re
from intervaltree import IntervalTree
import concurrent.futures
from functools import partial
import subprocess
import tempfile
import shutil
from datetime import datetime

In [ ]:
with open('ld_lists/params_ld_comparison.txt', 'r') as f:
    params_content = f.readline()
params = params_content.split(" ")
file_path_summary_stats = params[0]
directory_1000_genomes = params[1]
coding_snp_list_path = params[2]
track_list = params[3]
print(file_path_summary_stats, directory_1000_genomes, coding_snp_list_path, track_list)
ld_based = True
plink_path = params[4]
ld_reference_path = params[5]
alpha = params[6]
print(ld_based, plink_path, ld_reference_path, alpha)

In [ ]:
!plink

### Skip this by just loading from the csv file:

In [ ]:
# Takes about 2.5minutes
selected_columns = ['chr', 'pos', 'ref', 'alt', 'neglog10_pval_EUR']
file = gzip.open(file_path_summary_stats, "rt") if file_path_summary_stats.endswith(".tsv.bgz") else open(file_path_summary_stats, "r")
df_summary_stats = pd.concat(pd.read_csv(file, sep="\t", usecols=selected_columns, chunksize=100000), ignore_index=True)
df_summary_stats

In [ ]:
## STEP 1: Process summary statistics
# Reverse the -log(pvalue) operation and drop the original column
df_summary_stats['p_value'] = 10 ** (-df_summary_stats['neglog10_pval_EUR'])
df_summary_stats.drop(columns=['neglog10_pval_EUR'], inplace=True)
df_summary_stats.dropna(subset=['p_value'], inplace=True)
df_summary_stats.reset_index(drop=True, inplace=True)
df_summary_stats

In [ ]:
df_summary_stats.to_csv('ld_lists/sumstats_for_debug.csv')

### Load sumstats from csv file:

In [ ]:
df_summary_stats = pd.read_csv('ld_lists/sumstats_for_debug.csv')

In [ ]:
track_list_file = pd.read_csv(track_list, header=None)
track_list = track_list_file.iloc[:, 0].tolist()
track_list

In [ ]:
WINDOW_SIZE = 500000
R2_THRESHOLD = 0.2  
KB_RADIUS = 500     

In [ ]:
## STEP 2: Merge with SAD data
# Initialize SAD columns with track names
sad_columns = [f"SAD{track}" for track in track_list]
# Pre-group summary stats by chromosome for faster joins
summary_stats_by_chr = {
    chr_val: sub_df.copy().set_index(['chr', 'pos', 'ref', 'alt'])
    for chr_val, sub_df in df_summary_stats.groupby('chr')
}
# Extract chromosome from filename with regex
pattern = re.compile(r'\.MAF_threshold=0\.005\.(\d+)_combined\.csv')
csv_files = [f for f in os.listdir(directory_1000_genomes) if f.endswith('.csv') and f != 'targets.csv']

### Skip this if using external file:

In [ ]:
def process_csv_file(csv_file, directory_1000_genomes, sad_columns, pattern, summary_stats_by_chr):
    """
    Process a single CSV file:
      - Reads the CSV and extracts the chromosome using the precompiled regex pattern.
      - Retrieves the corresponding summary stats from summary_stats_by_chr.
      - Sets the index on the merge keys and performs a join.
      - Returns the merged DataFrame for that CSV.
    """
    csv_file_path = os.path.join(directory_1000_genomes, csv_file)
    try:
        chunk = pd.read_csv(csv_file_path, usecols=['chr', 'pos', 'ref', 'alt', 'snp'] + sad_columns)
    except Exception as e:
        print(f"Error reading {csv_file_path}: {e}", flush=True)
        return None

    match = pattern.search(csv_file)
    if not match:
        return None
    chromosome = int(match.group(1))
    
    df_summary_stats_chr = summary_stats_by_chr.get(chromosome)
    if df_summary_stats_chr is None or df_summary_stats_chr.empty:
        return None
    
    # Set index on chunk to match summary stats index
    chunk = chunk.set_index(['chr', 'pos', 'ref', 'alt'])
    merged_result = df_summary_stats_chr.join(chunk, how='left').reset_index()
    
    # Keep only rows with valid SAD and p_value values
    merged_result = merged_result[merged_result[sad_columns[0]].notna() & merged_result['p_value'].notna()]
    print(f"----Processed file {csv_file_path}", flush=True)
    return merged_result

In [ ]:
%%writefile tasks.py
def process_csv_file(csv_file, directory_1000_genomes, sad_columns, pattern, summary_stats_by_chr):
    """
    Process a single CSV file:
      - Reads the CSV and extracts the chromosome using the precompiled regex pattern.
      - Retrieves the corresponding summary stats from summary_stats_by_chr.
      - Sets the index on the merge keys and performs a join.
      - Returns the merged DataFrame for that CSV.
    """
    csv_file_path = os.path.join(directory_1000_genomes, csv_file)
    try:
        chunk = pd.read_csv(csv_file_path, usecols=['chr', 'pos', 'ref', 'alt', 'snp'] + sad_columns)
    except Exception as e:
        print(f"Error reading {csv_file_path}: {e}", flush=True)
        return None

    match = pattern.search(csv_file)
    if not match:
        return None
    chromosome = int(match.group(1))
    
    df_summary_stats_chr = summary_stats_by_chr.get(chromosome)
    if df_summary_stats_chr is None or df_summary_stats_chr.empty:
        return None
    
    # Set index on chunk to match summary stats index
    chunk = chunk.set_index(['chr', 'pos', 'ref', 'alt'])
    merged_result = df_summary_stats_chr.join(chunk, how='left').reset_index()
    
    # Keep only rows with valid SAD and p_value values
    merged_result = merged_result[merged_result[sad_columns[0]].notna() & merged_result['p_value'].notna()]
    print(f"----Processed file {csv_file_path}", flush=True)
    return merged_result

### Process csv files in parallel:

In [ ]:
from tasks import process_csv_file
# Process files in parallel
dataframes = []
with concurrent.futures.ProcessPoolExecutor() as executor:
    func = partial(process_csv_file,
                  directory_1000_genomes=directory_1000_genomes,
                  sad_columns=sad_columns,
                  pattern=pattern,
                  summary_stats_by_chr=summary_stats_by_chr)
    results = list(executor.map(func, csv_files))

In [ ]:
# Combine results
for res in results:
    if res is not None and not res.empty:
        dataframes.append(res)
    
df_summary_stats_result = pd.concat(dataframes, ignore_index=True)